# Predicting Hotel Booking Cancellations: Comparing Five Classification Models

**Course:** Machine Learning (M.Tech AIML / DSE), Assignment 2

**Problem statement:** A hotel loses revenue when a guest cancels at the last minute, because the room may go empty. If the hotel can *predict* which bookings are likely to be cancelled, it can plan overbooking, send reminders or ask for deposits. This notebook builds and compares **five classification models** that predict whether a booking will be cancelled (`is_canceled`: 1 = cancelled, 0 = not cancelled).

**Notebook roadmap (mirrors the assignment steps):**

| Step | What we do |
|---|---|
| 1 | Choose and understand the dataset (features, target, class distribution, missing values) |
| 2 | Preprocess: duplicates, missing values, encoding, scaling, stratified 80:20 split |
| 3 | Train 5 models and compute Accuracy, AUC, Precision, Recall, F1, MCC |
| 4 | Visualise results and save the models and test data for the GitHub repo |

## 0. Setup

We import the libraries and fix a **random seed**. The seed makes the train-test split and the model training repeatable: anyone who runs this notebook gets the same numbers.

> **Tip:** change `RANDOM_SEED` to a number of your own choice (for example your birth date such as `1507`). Then your results are yours and not identical to another student's.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ---- configuration -------------------------------------------------
RANDOM_SEED = 219       # my seed number (written 0219; Python does not allow a leading zero)
TEST_SIZE = 0.20        # 80:20 split as required
TARGET = "is_canceled"

# The notebook lives in <repo>/model/, so the repo root is one folder up.
ROOT = Path.cwd().parent if Path.cwd().name == "model" else Path.cwd()
MODEL_DIR = ROOT / "model"
RESULTS_DIR = ROOT / "results"
MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
np.random.seed(RANDOM_SEED)
print("Repo root:", ROOT)

## Step 1: Dataset choice

**Dataset:** *Hotel Booking Demand* (Kaggle: `hotel_bookings.csv`), originally published by Antonio, Almeida and Nunes (2019, *Data in Brief*). It contains real booking records of a city hotel and a resort hotel in Portugal.

**Why this dataset?**
- It satisfies every requirement: far more than **12 features** (31 predictors) and more than **1,000 instances** (about 119,000 rows).
- It has real **missing values** (`children`, `country`, `agent`, `company`) that we must handle.
- The target is **imbalanced** (more non-cancelled than cancelled bookings), so we must look at the class distribution and not rely on accuracy alone.
- It mixes **numeric and categorical** features, which gives a genuine preprocessing task (encoding and scaling).
- The problem is practical: predicting cancellations has direct business value.

**Target variable:** `is_canceled` (binary: 1 = booking was cancelled, 0 = booking was honoured).

In [ ]:
candidates = [ROOT / "hotel_bookings.csv",
              ROOT / "data" / "hotel_bookings.csv",
              Path.cwd() / "hotel_bookings.csv"]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("hotel_bookings.csv not found. Download it from Kaggle "
                            "(Hotel booking demand) and place it in the repo root or in a data/ folder.")

df_raw = pd.read_csv(DATA_PATH)
print("Loaded from:", DATA_PATH)
print("Rows, columns:", df_raw.shape)
df_raw.head()

### 1.1 Feature types

`pandas` tells us which columns are numbers and which are text (categories). This matters because the two kinds need different preprocessing: numbers are scaled, categories are one-hot encoded.

In [ ]:
feature_cols = [c for c in df_raw.columns if c != TARGET]
numeric_raw = df_raw[feature_cols].select_dtypes(include="number").columns.tolist()
categorical_raw = df_raw[feature_cols].select_dtypes(exclude="number").columns.tolist()

print(f"Total predictor features : {len(feature_cols)}")
print(f"Numeric features ({len(numeric_raw)}): {numeric_raw}\n")
print(f"Categorical features ({len(categorical_raw)}): {categorical_raw}")

### 1.2 Class distribution (class imbalance check)

If one class is much rarer than the other, a model can look good on **accuracy** by simply predicting the majority class. That is why the assignment also asks for AUC, Precision, Recall, F1 and MCC.

In [ ]:
counts = df_raw[TARGET].value_counts().sort_index()
percent = (df_raw[TARGET].value_counts(normalize=True).sort_index() * 100).round(2)
dist = pd.DataFrame({"count": counts, "percent": percent})
dist.index = ["Not cancelled (0)", "Cancelled (1)"]
display(dist)

ratio = counts.max() / counts.min()
print(f"Majority : minority ratio = {ratio:.2f} : 1")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x=TARGET, data=df_raw, ax=ax)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Not cancelled (0)", "Cancelled (1)"])
ax.set_title("Class distribution of the target")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=150)
plt.show()

**Reading the result:** the data is moderately imbalanced (roughly 63% / 37%). It is not extreme, so we do not need resampling, but we will (a) keep the same proportions in train and test using a *stratified* split and (b) judge the models on F1, recall and MCC and not only on accuracy.

### 1.3 Missing values (identify)

In [ ]:
missing = df_raw.isnull().sum()
missing_df = pd.DataFrame({"missing_count": missing,
                           "missing_percent": (missing / len(df_raw) * 100).round(2)})
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values("missing_count", ascending=False)
display(missing_df)

**What the missing values mean:**
- `company` is missing for about 94% of rows. A column that empty carries almost no information, so we **drop it**.
- `agent` is missing for about 14% of rows. A missing agent means the booking was made **without a travel agent**, so the blank is meaningful. We keep it as its own category called `Missing`.
- `country` has a few hundred blanks. We label them `Missing` as well.
- `children` has only a handful of blanks. We fill them with the **median** (learned from the training data only).

Also note that some text columns contain the word `Undefined` (for example in `meal`). That is a legitimate category in this dataset, so we leave it as it is.

## Step 2: Data preprocessing

The golden rule: **anything that is *learned* from data (medians, means, category lists, scaling factors) must be learned from the training set only.** If the test set influences these, information "leaks" and the scores look better than they really are. We therefore:

1. Do the cleaning that needs no learning (duplicates, leakage columns, impossible rows) on the whole table.
2. Split into train and test.
3. Fit imputers, scaler and encoder on the **training data only** and then apply them to the test data.

### 2.1 Handle duplicates

In [ ]:
n_dup = df_raw.duplicated().sum()
print(f"Duplicate rows found: {n_dup:,} ({n_dup / len(df_raw) * 100:.1f}% of the data)")

df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Rows after removing duplicates: {len(df):,}")

Duplicates are removed *before* splitting. Otherwise the same booking could land in both the train and the test set, and the model would be tested on rows it has already seen.

### 2.2 Remove data-leakage columns

`reservation_status` (Check-Out / Canceled / No-Show) and `reservation_status_date` are only known **after** the outcome is decided. Using them would be cheating, because the model could read the answer from them. Let us prove it:

In [ ]:
display(pd.crosstab(df["reservation_status"], df[TARGET]))

`Canceled` always equals `is_canceled = 1`, so this column is basically the target in disguise. We drop both leakage columns, and also `company` (94% missing, see 1.3).

In [ ]:
LEAKAGE_COLS = ["reservation_status", "reservation_status_date"]
HIGH_MISSING_COLS = ["company"]
df = df.drop(columns=LEAKAGE_COLS + HIGH_MISSING_COLS)
print("Columns left:", df.shape[1])

### 2.3 Remove impossible rows and tidy the `agent` column

- A booking with **zero guests** (no adults, children or babies) is a data error, so we remove those rows. This is a fixed rule and learns nothing from the data, so it is safe to do before the split.
- `agent` is an ID number. IDs are *labels*, not quantities (agent 20 is not "twice" agent 10), so we convert it to text to treat it as a **category**.

In [ ]:
total_guests = df["adults"] + df["children"].fillna(0) + df["babies"]
print(f"Bookings with zero guests removed: {(total_guests == 0).sum()}")
df = df[total_guests > 0].reset_index(drop=True)

df["agent"] = df["agent"].map(lambda v: f"agent_{int(v)}" if pd.notna(v) else np.nan)
print("Rows now:", len(df))

### 2.4 Train-test split (80:20, stratified, fixed seed)

`stratify=y` keeps the proportion of cancelled / not-cancelled bookings the same in both parts. We verify this below.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")
split_check = pd.DataFrame({
    "Full data (%)": (y.value_counts(normalize=True) * 100).round(2),
    "Train (%)": (y_train.value_counts(normalize=True) * 100).round(2),
    "Test (%)": (y_test.value_counts(normalize=True) * 100).round(2)})
display(split_check)

### 2.5 Encoding, scaling and missing-value handling (fitted on training data only)

We describe the preprocessing once, as a `ColumnTransformer`:

| Column type | Missing values | Transformation | Why |
|---|---|---|---|
| Numeric | Median of the training data | `StandardScaler` (mean 0, std 1) | kNN and Logistic Regression are sensitive to feature scale |
| Categorical | Replaced by the label `Missing` | `OneHotEncoder` (rare categories under 1% grouped together) | Models need numbers, and grouping rare categories (e.g. obscure countries) keeps the feature count manageable |

Wrapping this in a `Pipeline` together with each model guarantees that `fit` only ever sees the training data.

In [ ]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
print(f"{len(num_cols)} numeric columns, {len(cat_cols)} categorical columns")

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                             min_frequency=0.01, sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

# Quick demonstration on a copy: fit on TRAIN only, then transform both.
demo = clone(preprocessor)
Xtr_demo = demo.fit_transform(X_train)
Xte_demo = demo.transform(X_test)
print("Processed train shape:", Xtr_demo.shape)
print("Processed test shape :", Xte_demo.shape)
print("Missing values left in train / test:", int(np.isnan(Xtr_demo).sum()), "/", int(np.isnan(Xte_demo).sum()))

## Step 3: Classification models and evaluation metrics

All five models are trained on the **same** training data and evaluated on the **same** test data.

| Model | Key settings | Idea in one line |
|---|---|---|
| Logistic Regression | `max_iter=1000` | Draws a straight-line (linear) boundary and outputs a probability |
| Decision Tree | `max_depth=10`, `min_samples_leaf=20` | A flow-chart of yes/no questions; depth is limited to avoid memorising the training data |
| kNN | `n_neighbors=7` | Looks at the 7 most similar bookings and takes a vote |
| Naive Bayes (Gaussian) | defaults | Uses Bayes' rule and assumes features are independent |
| Random Forest | `n_estimators=100`, `max_depth=18`, `min_samples_leaf=5` | Many decision trees voting together (ensemble) |

**Metrics** (the positive class is *cancelled = 1*):
- **Accuracy:** share of all predictions that are correct.
- **AUC:** how well the model *ranks* cancellations above non-cancellations (0.5 = guessing, 1.0 = perfect).
- **Precision:** of the bookings predicted as cancelled, how many really were.
- **Recall:** of the bookings that really were cancelled, how many we caught.
- **F1:** the balance (harmonic mean) of precision and recall.
- **MCC (Matthews Correlation Coefficient):** one balanced score from -1 to +1 that stays honest on imbalanced data.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=RANDOM_SEED),
    "kNN": KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(n_estimators=100, max_depth=18,
                                                        min_samples_leaf=5, n_jobs=-1,
                                                        random_state=RANDOM_SEED),
}

def evaluate(y_true, y_pred, y_score):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_score),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }

fitted_pipes, predictions, rows = {}, {}, {}

for name, clf in models.items():
    pipe = Pipeline([("prep", clone(preprocessor)), ("clf", clf)])
    pipe.fit(X_train, y_train)                 # preprocessing is fitted on TRAIN only
    y_pred = pipe.predict(X_test)
    y_score = pipe.predict_proba(X_test)[:, 1]
    fitted_pipes[name] = pipe
    predictions[name] = (y_pred, y_score)
    rows[name] = evaluate(y_test, y_pred, y_score)
    print(f"done: {name}")

results_df = pd.DataFrame(rows).T
results_df.index.name = "ML Model Name"
results_df = results_df.round(4)
results_df.to_csv(RESULTS_DIR / "model_comparison.csv")
display(results_df)

In [ ]:
# Which model is best on each metric?
best = results_df.idxmax()
display(pd.DataFrame({"Best model": best, "Score": results_df.max()}))

## Step 4: Visualising the results

### 4.1 All metrics for all models (bar chart)

In [ ]:
plot_df = results_df.reset_index().melt(id_vars="ML Model Name", var_name="Metric", value_name="Score")

fig, ax = plt.subplots(figsize=(12, 5.5))
sns.barplot(data=plot_df, x="Metric", y="Score", hue="ML Model Name", ax=ax)
ax.set_title("Evaluation metrics of all models (test set)")
ax.set_ylim(0, 1.05)
ax.legend(title="Model", loc="lower center", bbox_to_anchor=(0.5, -0.38), ncol=5)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "metrics_comparison_bar.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.2 Heatmap of the comparison table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
sns.heatmap(results_df, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0, vmax=1, ax=ax)
ax.set_title("Model vs metric")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "metrics_heatmap.png", dpi=150)
plt.show()

### 4.3 ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for name, (_, y_score) in predictions.items():
    fpr, tpr, _ = roc_curve(y_test, y_score)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {results_df.loc[name, 'AUC']:.3f})")
ax.plot([0, 1], [0, 1], "k--", label="Random guess")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "roc_curves.png", dpi=150)
plt.show()

### 4.4 Confusion matrices

Each box counts predictions: the diagonal is correct, the off-diagonal boxes are mistakes (missed cancellations and false alarms).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, (y_pred, _)) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Kept", "Cancelled"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrices.png", dpi=150)
plt.show()

### 4.5 What drives cancellations? (Random Forest feature importance)

This helps explain *why* the best model works and gives useful material for your observations.

In [ ]:
rf_pipe = fitted_pipes["Random Forest (Ensemble)"]
feat_names = rf_pipe.named_steps["prep"].get_feature_names_out()
importances = pd.Series(rf_pipe.named_steps["clf"].feature_importances_, index=feat_names)
top = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5.5))
sns.barplot(x=top.values, y=top.index, ax=ax, color="#3b7dd8")
ax.set_title("Top 15 features (Random Forest)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "rf_feature_importance.png", dpi=150)
plt.show()

## Saving models and test data (for the GitHub repository)

The assignment requires saved model files and a `test_data.csv`. Each saved file is a full pipeline (preprocessing and model), so it can predict directly from raw booking rows.

In [ ]:
file_names = {}
for name, pipe in fitted_pipes.items():
    fname = (name.lower().replace(" (ensemble)", "").replace(" ", "_")) + ".pkl"
    joblib.dump(pipe, MODEL_DIR / fname, compress=3)
    file_names[name] = fname

# Test data: the raw (un-preprocessed) 20% hold-out plus the true label
test_data = X_test.copy()
test_data[TARGET] = y_test.values
test_data.to_csv(ROOT / "test_data.csv", index=False)

for name, fname in file_names.items():
    size_mb = (MODEL_DIR / fname).stat().st_size / 1e6
    flag = "  <-- WARNING: GitHub rejects files over 100 MB" if size_mb > 95 else ""
    print(f"{fname:<28}{size_mb:8.2f} MB{flag}")
print(f"test_data.csv rows: {len(test_data):,}")

### Check: reload a saved model and score `test_data.csv`

If this reproduces the Random Forest accuracy above, the saved files are working correctly.

In [ ]:
reloaded = joblib.load(MODEL_DIR / file_names["Random Forest (Ensemble)"])
check = pd.read_csv(ROOT / "test_data.csv")
acc = accuracy_score(check[TARGET], reloaded.predict(check.drop(columns=[TARGET])))
print(f"Reloaded Random Forest accuracy on test_data.csv: {acc:.4f}")
print(f"Accuracy in results table                       : {results_df.loc['Random Forest (Ensemble)', 'Accuracy']:.4f}")

## Observations

*(To be completed after the final run: copy your own words into the README.)*

| ML Model Name | Observation about model performance |
|---|---|
| Logistic Regression | |
| Decision Tree | |
| kNN | |
| Naive Bayes | |
| Random Forest (Ensemble) | |
| **Overall winner for this dataset** | |